# SafeWalk: export `best.pt` for Coral Edge TPU

Run this notebook in Google Colab. Edge TPU export requires x86 Linux and a representative dataset for INT8 calibration.

In [ ]:
!pip install -q ultralytics==8.4.104

import platform
from pathlib import Path

print("Machine:", platform.machine())
if platform.machine() not in {"x86_64", "AMD64"}:
    raise RuntimeError("Edge TPU export must run on x86 Linux. Use Google Colab.")

## Upload the trained checkpoint

In [ ]:
from google.colab import files

uploaded = files.upload()
checkpoints = [Path(name) for name in uploaded if name.lower().endswith(".pt")]
if len(checkpoints) != 1:
    raise RuntimeError("Upload exactly one .pt model file.")

MODEL_PATH = checkpoints[0]
print("Using checkpoint:", MODEL_PATH)

## Locate the ROD calibration dataset

Run your dataset download/setup cells first if this path does not exist. The images referenced by `data.yaml` must also be present.

In [ ]:
DATA_YAML = Path("/content/obstacle_dataset/ROD-Dataset/dataset/data.yaml")

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Dataset config not found: {DATA_YAML}. "
        "Run the SafeWalk dataset setup cells before exporting."
    )

print("Calibration dataset:", DATA_YAML)

## Export and compile

The first export uses 1% of the dataset for calibration and a 320-pixel input to fit within standard Colab RAM. Save the compiler output for your report.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
exported_path = model.export(
    format="edgetpu",
    imgsz=320,
    quantize=8,
    data=str(DATA_YAML),
    fraction=0.01,
    device="cpu",
)

print("Export result:", exported_path)

## Find and download the compiled model

In [ ]:
import shutil
from google.colab import files

candidates = sorted(Path("/content").rglob("*_edgetpu.tflite"))
if not candidates:
    raise FileNotFoundError(
        "No *_edgetpu.tflite file was created. Review the export/compiler output above."
    )

for candidate in candidates:
    print(candidate)

final_model = Path("/content/safewalk_edgetpu.tflite")
shutil.copy2(candidates[-1], final_model)
print("Downloading:", final_model)
files.download(str(final_model))